In [1]:
from pathlib import Path
import numpy as np
import os, json, random, pickle
from collections import Counter
from pathlib import Path

In [2]:
# Check image path

omama_dir_path = "/raid/mpsych/OMAMA/DATA/data/2d_resized_1024/images"
if not os.path.exists(omama_dir_path):
    print(f"File not found: {omama_dir_path}")
    print("Please update dicom_path with a valid image file path")
else:
    print(f"Found DICOM folder: {os.path.basename(omama_dir_path)}")
    omama_folder = Path("/raid/mpsych/OMAMA/DATA/data/2d_resized_1024/images")
    IDs = sorted(str(file) for file in omama_folder.rglob("*.npz"))
    print(len(IDs))

Found DICOM folder: images
163568


In [3]:
meta_dir = Path("/hpcstor6/scratch01/a/anya.tongprasith001/U54REC/omama/2d_resized_256/metadata")

In [4]:
# Get the headers
pkl = Path("/home/anya.tongprasith001/U54REC/release_to_header_mapping.pkl")
mapping = pickle.load(open(pkl, "rb"))
len(mapping)

163568

In [5]:
# Get the images, labels, and headers
noncancer = {}
cancer = {}

for ID in IDs :
    meta = json.load(open(meta_dir / f"{Path(ID).stem}.json"))
    if meta["label"] == "Unknown" :
        continue
        
    # Load image
    img_name = ID
    # Load window values
    wc = meta["WindowCenter"]
    ww = meta["WindowWidth"]
    window_center = float(wc[0] if hasattr(wc, '__len__') else wc)
    window_width = float(ww[0] if hasattr(ww, '__len__') else ww)
    window_min = window_center - window_width / 2
    window_max = window_center + window_width / 2

    # Load headers
    ds = mapping[Path(ID).stem]
    age_str = ds.PatientAge
    if age_str and age_str != '':
        age = float(age_str[:-1]) / 12.0
    else:
        age = -1.0 
    breastimplant = ds.get("BreastImplantPresent")
    if breastimplant == "None" :
        breastimplant = "NO"
    view = ds.ViewPosition
    if meta["label"] == "NonCancer" :
        if meta["PatientID"] not in noncancer.keys() :
            noncancer[meta["PatientID"]] = []
        noncancer[meta["PatientID"]].append([img_name, [window_min, window_max], 0, 
                                             [age, breastimplant, view
                                             ]])

    else : 
        if meta["PatientID"] not in cancer.keys() :
            cancer[meta["PatientID"]] = []
        cancer[meta["PatientID"]].append([img_name, [window_min, window_max], 1, 
                                          [age, breastimplant, view
                                          ]])
        

In [7]:
nc = list(noncancer.items())
c = list(cancer.items())

print(len(nc), len(c))

154238 3562


In [13]:
# Randomized the patients
random.shuffle(c)
random.shuffle(nc)

In [16]:
# Select cancer patients for test data
test_amount = int(0.15 * len(c))
test_c = c[:test_amount]
trainval_c = c[test_amount:]
print(len(test_c), len(trainval_c))

534 3028


In [32]:
# Assign the images, metadata, and labels for setting up mix input in cancer cases
def assign_data(dataset) :
    all_patient = []
    for patient, data_list in dataset :
        for data in data_list :
            img = data[0]
            metadata = data[1]
            label = data[2]
            headers = data[3]
            all_patient.append((img, metadata, label, headers))
    return all_patient

test_c_data = assign_data(test_c)

In [33]:
# Get same numbers of images for non-cancer cases
test_nc = nc[:len(test_c_data)]
trainval_nc = nc[len(test_c_data):]
test_nc_data = assign_data(test_nc)
print(len(test_c_data), len(test_nc_data))

1109 1109


In [34]:
# Set up numpy arrays
def np_setup(dataset_c, dataset_nc) :
    imgs, metadata, labels, headers_num, headers_str = [],[],[],[],[]
    
    # Merge cancer and non-cancer, shuffle
    dataset_all = dataset_c + dataset_nc
    random.shuffle(dataset_all)
    
    for data in dataset_all :
        imgs.append(data[0])
        metadata.append(data[1])
        labels.append(data[2])
        headers_num.append(data[3][0])
        headers_str.append((data[3][0:]))
    imgs_np = np.array(imgs)
    metadata_np = np.array(metadata, dtype=np.float32)
    labels_np = np.array(labels, dtype=np.float32)
    headers_num_np = np.array(headers_num, dtype=np.float32)
    headers_str_np = np.array(headers_str, dtype=str)
    return imgs_np, metadata_np, labels_np, headers_num_np, headers_str_np

In [35]:
test_imgs, test_metadata, test_labels, test_headers_num, test_headers_str = np_setup(test_c_data, test_nc_data)